# Passo a passo: como o programa "enxerga" uma fruta

Este notebook mostra cada etapa com imagens. Use para **entender**, para **conferir as manchas**
e para **tirar figuras para o relatório**.

Rode as células em ordem (Shift+Enter). Antes, rode `python src/baixar_fotos.py`.

In [ ]:
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt

sys.path.append(str(Path('..', 'src').resolve()))
from preprocessamento import preprocessar
from medidas import converter_hsv, mascara_manchas, percentual_manchas

def mostrar_rgb(ax, imagem_bgr, titulo):
    # O OpenCV guarda as cores em BGR; o matplotlib espera RGB.
    ax.imshow(cv2.cvtColor(imagem_bgr, cv2.COLOR_BGR2RGB))
    ax.set_title(titulo)
    ax.axis('off')

## 1. Escolher uma foto
Troque o número para ver outras fotos.

In [ ]:
fotos = sorted(f for f in Path('..', 'dataset').glob('*/*/*.*') if f.suffix.lower() in {'.jpg', '.jpeg', '.png'})
print(len(fotos), 'fotos encontradas')
caminho = fotos[0]   # <- troque o número
print('Usando:', caminho)

## 2. Separar a fruta do fundo
1. Converte para **HSV** (H = cor, S = quão viva, V = quão clara)
2. A fruta é mais colorida (canal **S** alto) que o fundo
3. Cria a **máscara**: branco = fruta, preto = fundo

In [ ]:
imagem, mascara, contorno = preprocessar(caminho)
hsv = converter_hsv(imagem)

fig, axs = plt.subplots(1, 4, figsize=(16, 4))
mostrar_rgb(axs[0], imagem, 'Foto')
axs[1].imshow(hsv[:, :, 1], cmap='gray'); axs[1].set_title('Canal S (saturação)'); axs[1].axis('off')
axs[2].imshow(mascara, cmap='gray'); axs[2].set_title('Máscara'); axs[2].axis('off')
mostrar_rgb(axs[3], cv2.bitwise_and(imagem, imagem, mask=mascara), 'Só a fruta')
plt.show()

## 3. Encontrar as manchas
Partes **escuras**, **marrons** ou **brancas (mofo)** dentro da fruta, usando `cv2.inRange`.
Se a máscara de manchas marcar coisa errada, ajustem `FAIXAS_MANCHA` em `src/medidas.py`.

In [ ]:
manchas = mascara_manchas(hsv, mascara)
fig, axs = plt.subplots(1, 2, figsize=(8, 4))
mostrar_rgb(axs[0], imagem, 'Foto')
axs[1].imshow(manchas, cmap='gray'); axs[1].set_title('Manchas encontradas'); axs[1].axis('off')
plt.show()
print(f'{percentual_manchas(hsv, mascara):.0%} da fruta tem manchas')

## 4. A "impressão digital de cor"
Quantos pixels da fruta têm cada cor. É isso que o programa compara para saber qual é a fruta.

In [ ]:
hist_h = cv2.calcHist([hsv], [0], mascara, [180], [0, 180])
plt.figure(figsize=(8, 3))
plt.plot(hist_h, color='darkorange')
plt.xlabel('cor (H): 0 = vermelho, 30 = amarelo, 60 = verde, 130 = roxo')
plt.ylabel('pixels')
plt.show()

## 5. Várias fotos lado a lado
Bom para o relatório: mostra a separação da fruta funcionando (ou falhando).

In [ ]:
amostra = fotos[::max(1, len(fotos) // 6)][:6]
fig, axs = plt.subplots(2, len(amostra), figsize=(3 * len(amostra), 6), squeeze=False)
for i, f in enumerate(amostra):
    img, msk, _ = preprocessar(f)
    mostrar_rgb(axs[0, i], img, f'{f.parent.parent.name}/{f.parent.name}')
    axs[1, i].imshow(mascara_manchas(converter_hsv(img), msk), cmap='gray'); axs[1, i].axis('off')
plt.show()